# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [ ]:
import io
import re
import sys
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
from playwright.sync_api import TimeoutError as PWTimeout
from playwright.sync_api import sync_playwright

In [ ]:
URL_PAGINA = "https://amefibra.com/el-mercado/indice-fibras/"
PATRON_IFRAME_TABLA = re.compile(r"edimex\.com\.mx/Emisora/Reportes/?(\?.*)?$")
CARPETA_SALIDA = Path.cwd() / "output"
FUENTE_DATOS = "AMEFIBRA / Economatica México"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Funciones de extracción

In [ ]:
def _limpiar_encabezado(texto: str) -> str:
    texto = re.sub(r"[↑↓]", "", str(texto))
    return re.sub(r"\s+", " ", texto).strip()


def _localizar_frame_tabla(page, intentos=10, espera_ms=1000):
    for _ in range(intentos):
        for frame in page.frames:
            if PATRON_IFRAME_TABLA.search(frame.url or ""):
                return frame
        page.wait_for_timeout(espera_ms)
    return None


def _extraer_html_tabla(frame) -> Optional[str]:
    return frame.evaluate("""() => {
        const tablas = Array.from(document.querySelectorAll('table'));
        let mejor = null, filasMax = -1;
        for (const tabla of tablas) {
            const filas = tabla.querySelectorAll('tbody tr').length;
            if (filas > filasMax) { filasMax = filas; mejor = tabla; }
        }
        return mejor ? mejor.outerHTML : null;
    }""")


def obtener_tabla_fibras(headless: bool = True, timeout_datos_ms: int = 30000) -> pd.DataFrame:
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=headless)
        page = browser.new_page(locale="es-MX")
        try:
            page.goto(URL_PAGINA, wait_until="domcontentloaded")
            frame = _localizar_frame_tabla(page)
            if frame is None:
                raise RuntimeError("No se encontró el iframe con la tabla de FIBRAs.")
            try:
                frame.wait_for_function("""() => {
                    const filas = document.querySelectorAll('table tbody tr');
                    if (filas.length === 0) return false;
                    const celda = filas[0].querySelector('td:nth-child(2)');
                    const texto = celda ? celda.textContent.trim() : '';
                    return texto.length > 0 && texto !== '0' && texto !== '0.00';
                }""", timeout=timeout_datos_ms)
            except PWTimeout:
                print("Aviso: se agotó el tiempo esperando datos en vivo; se usará lo cargado.", file=sys.stderr)
            html_tabla = _extraer_html_tabla(frame)
        finally:
            browser.close()
    if not html_tabla:
        raise RuntimeError("No se pudo extraer la tabla de indicadores.")
    df = pd.read_html(io.StringIO(html_tabla))[0]
    df.columns = [_limpiar_encabezado(columna) for columna in df.columns]
    return df.dropna(axis=1, how="all")

## Normalización y exportación

In [ ]:
def _a_snake_case(texto: str) -> str:
    texto = texto.replace("%", "pct")
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", "_", sin_acentos).strip("_").lower()


def normalizar_para_analisis(df: pd.DataFrame, momento_extraccion: Optional[datetime] = None) -> pd.DataFrame:
    momento_extraccion = momento_extraccion or datetime.now()
    resultado = df.copy()
    resultado.columns = [_a_snake_case(str(columna)) for columna in resultado.columns]
    for columna in resultado.columns:
        serie = resultado[columna]
        if pd.api.types.is_string_dtype(serie):
            valores = serie.astype(str).str.strip()
            if valores.str.endswith("%").all():
                resultado[columna] = pd.to_numeric(valores.str.rstrip("%"), errors="coerce")
    resultado.insert(0, "fecha_hora_extraccion", momento_extraccion.isoformat(timespec="seconds"))
    resultado.insert(1, "fuente_datos", FUENTE_DATOS)
    return resultado


def exportar_csv_analitico(df: pd.DataFrame, carpeta_salida: Path = CARPETA_SALIDA) -> Path:
    momento = datetime.now()
    carpeta_salida.mkdir(parents=True, exist_ok=True)
    nombre = f"{momento:%Y%m%d_%H%M%S}_indice_fibras_amefibra.csv"
    ruta = carpeta_salida / nombre
    normalizar_para_analisis(df, momento).to_csv(ruta, index=False, encoding="utf-8")
    return ruta

## Ejecutar extracción

In [ ]:
print(f"Consultando {URL_PAGINA} ...")
df = obtener_tabla_fibras(headless=HEADLESS, timeout_datos_ms=TIMEOUT_DATOS_MS)
print(f"Índice FIBRAS - {datetime.now():%Y-%m-%d %H:%M} (dato con ~20 min de retraso)")
display(df)

if EXPORTAR_CSV_ANALITICO:
    ruta_csv_analitico = exportar_csv_analitico(df)
    print(f"CSV analítico guardado en: {ruta_csv_analitico}")

In [ ]:
if EXPORTAR_CSV_EXCEL:
    df.to_csv(RUTA_CSV_EXCEL, index=False, encoding="utf-8-sig")
    print(f"CSV compatible con Excel guardado en: {RUTA_CSV_EXCEL}")

if EXPORTAR_XLSX:
    df.to_excel(RUTA_XLSX, index=False)
    print(f"Excel guardado en: {RUTA_XLSX}")